In [2]:
import sqlite3
import pandas as pd
import rasterio
from rasterio.warp import transform_bounds
import json
import os

# --- 1. CONFIGURATION ---
# Replace these with your actual file paths
GEOTIFF_PATH = 'geotiff.tif'
SQLITE_DB_PATH = 'defid2.sqlite'
SPATIALITE_PATH = 'mod_spatialite' # Ensure this is in your PATH or update to absolute path

# --- 2. EXTRACT COORDINATES FROM YOUR GEOTIFF ---
# We use this as our "search light" to find the right rows in the database
with rasterio.open(GEOTIFF_PATH) as src:
    bounds = src.bounds
    # Transform coordinates to Latitude/Longitude (WGS84)
    wgs84_bounds = transform_bounds(src.crs, 'EPSG:4326', *bounds)
    min_x, min_y, max_x, max_y = wgs84_bounds
    center_lat = (min_y + max_y) / 2
    center_lon = (min_x + max_x) / 2

print(f"GeoTIFF Bounding Box (WGS84): {wgs84_bounds}")
print(f"Target Center Coordinate: {center_lat}, {center_lon}")

# --- 3. DATABASE CONNECTION ---
conn = sqlite3.connect(SQLITE_DB_PATH)

# Enable SpatiaLite extension
conn.enable_load_extension(True)
try:
    conn.load_extension(SPATIALITE_PATH)
    print("SpatiaLite extension loaded successfully.")
except Exception as e:
    print(f"Warning: Could not load SpatiaLite extension from '{SPATIALITE_PATH}'.")
    print("Specific error:", e)
    print("Make sure it is in your PATH or update the SPATIALITE_PATH variable.")

# --- 4. MATCHING THE GEOTIFF TO THE DATABASE ---
# We search for events intersecting the bounding box of our GeoTIFF.
try:
    query = f"""
    SELECT
        e.id AS event_id,
        e.survey_date,
        e.dataset_id,
        e.country_id,
        g.id AS geom_id
    FROM event e
    JOIN geoms_events ge ON e.id = ge.event_id
    JOIN geom g ON ge.geom_id = g.id
    WHERE MBRIntersects(g.the_geom, BuildMBR(
        {min_x}, {min_y}, {max_x}, {max_y}, 4326
    )) == 1;
    """
    df_matches = pd.read_sql_query(query, conn)
    
    if df_matches.empty:
        print("No matching events found within the GeoTIFF bounding box.")
    else:
        print(f"Found {len(df_matches)} matching event(s).")
        display(df_matches.head())
except Exception as e:
    print(f"Error querying database: {e}")
    df_matches = pd.DataFrame() # Ensure variable is defined

# --- 5. GENERATE WINDOWS FOR RSLEARN ---
# This creates the instructions for rslearn to go fetch the imagery and make embeddings
windows = []

if not df_matches.empty:
    for _, row in df_matches.iterrows():
        event_id = row['event_id']
        # Extract year from survey_date (assuming format like 'YYYY-MM-DD' or starting with YYYY)
        survey_date_str = str(row['survey_date'])
        try:
            start_year = int(survey_date_str[:4])
        except ValueError:
            start_year = 2023 # Fallback
        
        # We create TWO windows: One for the Infested year and one for the Year Before (Healthy)
        for state in ['healthy', 'infested']:
            target_year = start_year - 1 if state == 'healthy' else start_year
            
            # We pick mid-summer (June) for bark beetle detection
            timestamp = f"{target_year}-06-15T00:00:00Z"
            
            window = {
                "window_id": f"event_{event_id}_{state}",
                "center": [center_lat, center_lon], # Center of the GeoTIFF patch
                "time": timestamp,
                "label": state 
            }
            windows.append(window)

    # Save the configuration for rslearn
    with open('windows.json', 'w') as f:
        json.dump(windows, f, indent=4)
    print(f"\nMISSION COMPLETE: 'windows.json' generated with {len(windows)} entries.")
else:
    print("Skipping window generation as no matches were found.")

conn.close()

GeoTIFF Bounding Box (WGS84): (16.3707633067316, 49.45582812296804, 16.43339234803625, 49.495447098632305)
Target Center Coordinate: 49.47563761080018, 16.402077827383927
SpatiaLite extension loaded successfully.
Found 698 matching event(s).


,event_id,survey_date,dataset_id,country_id,geom_id
0,59285,2018-09-30T00:00:00,18,12,80505
1,428454,2021-09-30T00:00:00,18,12,449999
2,428455,2021-09-30T00:00:00,18,12,450000
3,271660,2019-09-30T00:00:00,18,12,292919
4,59248,2018-09-30T00:00:00,18,12,80468



MISSION COMPLETE: 'windows.json' generated with 1396 entries.


In [3]:
# --- [NEW] DIAGNOSTIC ANALYSIS OF THE AREA ---
# This cell provides a statistical deep dive into the observations within the bounding box.

try:
    conn = sqlite3.connect(SQLITE_DB_PATH)
    conn.enable_load_extension(True)
    conn.load_extension(SPATIALITE_PATH)
    
    # We query IDs, Dates, and Centroids (X/Y coordinates of each spot)
    diag_query = f"""
    SELECT
        e.id AS event_id,
        e.survey_date,
        X(ST_Centroid(g.the_geom)) AS lon,
        Y(ST_Centroid(g.the_geom)) AS lat
    FROM event e
    JOIN geoms_events ge ON e.id = ge.event_id
    JOIN geom g ON ge.geom_id = g.id
    WHERE MBRIntersects(g.the_geom, BuildMBR(
        {min_x}, {min_y}, {max_x}, {max_y}, 4326
    )) == 1;
    """
    
    df_diag = pd.read_sql_query(diag_query, conn)
    
    if not df_diag.empty:
        print("--- AREA OBSERVATION STATISTICS ---")
        print(f"Total Observations (Rows): {len(df_diag)}")
        print(f"Unique Event IDs:         {df_diag['event_id'].nunique()}")
        print(f"Unique Survey Dates:      {df_diag['survey_date'].nunique()}")
        
        # We group by lon/lat to find unique physical spots
        unique_coords = df_diag.groupby(['lon', 'lat']).size().reset_index().shape[0]
        print(f"Unique Physical Centroids: {unique_coords}")
        
        print("\n--- PREVIEW OF OBSERVATIONS (FIRST 20) ---")
        display(df_diag.head(20))
    else:
        print("No observations found in this bounding box.")
        
except Exception as e:
    print(f"Error during diagnostic analysis: {e}")
finally:
    if 'conn' in locals() and conn:
        conn.close()

--- AREA OBSERVATION STATISTICS ---
Total Observations (Rows): 698
Unique Event IDs:         697
Unique Survey Dates:      4
Unique Physical Centroids: 698

--- PREVIEW OF OBSERVATIONS (FIRST 20) ---


,event_id,survey_date,lon,lat
0,59285,2018-09-30T00:00:00,16.412254,49.491044
1,428454,2021-09-30T00:00:00,16.399305,49.467418
2,428455,2021-09-30T00:00:00,16.396752,49.467319
3,271660,2019-09-30T00:00:00,16.402945,49.482078
4,59248,2018-09-30T00:00:00,16.380630,49.481326
5,59250,2018-09-30T00:00:00,16.376756,49.479022
6,59251,2018-09-30T00:00:00,16.395762,49.478838
7,59252,2018-09-30T00:00:00,16.399689,49.480194
8,147144,2019-09-30T00:00:00,16.388320,49.468770
9,428420,2021-09-30T00:00:00,16.406203,49.463168
